# S50_01 — Quantization

Quantization reduces model size and inference speed by storing weights in lower-precision formats. It's the most important technique for deploying LLMs on consumer hardware.

## Precision formats

| Format | Bits | Values | Memory (7B model) | Use case |
|--------|------|--------|-------------------|----------|
| FP32 | 32 | ±3.4×10³⁸ | 28 GB | Training (rarely) |
| BF16 | 16 | ±3.4×10³⁸ (less precision) | 14 GB | Training standard |
| FP16 | 16 | ±65504 | 14 GB | Inference on GPU |
| INT8 | 8 | -128 to 127 | 7 GB | Inference, minor quality loss |
| NF4 (4-bit) | 4 | 16 values | ~4 GB | QLoRA training |
| GGUF Q4_K_M | ~4 | Mixed | ~4 GB | llama.cpp / Ollama |
| GGUF Q2_K | ~2 | Mixed | ~2 GB | Very low memory, quality hit |

In [ ]:
import numpy as np

# Demonstrate quantization: float32 → int8
def quantize_int8(weights):
    """Symmetric per-tensor INT8 quantization."""
    scale = np.max(np.abs(weights)) / 127
    quantized = np.clip(np.round(weights / scale), -127, 127).astype(np.int8)
    return quantized, scale

def dequantize_int8(quantized, scale):
    return quantized.astype(np.float32) * scale

# Simulate a small weight matrix
np.random.seed(42)
weights = np.random.randn(4, 4).astype(np.float32)

quantized, scale = quantize_int8(weights)
reconstructed = dequantize_int8(quantized, scale)

print('Original weights (FP32):')
print(weights.round(4))
print('\nQuantized (INT8):')
print(quantized)
print(f'Scale: {scale:.6f}')
print('\nReconstructed:')
print(reconstructed.round(4))
print(f'\nMax reconstruction error: {np.max(np.abs(weights - reconstructed)):.6f}')
print(f'Memory reduction: 4x (32-bit → 8-bit)')

## GGUF quantization formats (llama.cpp)

In [ ]:
# GGUF is the format used by llama.cpp, Ollama, and LM Studio
# K-quants use mixed precision: some layers stay at higher precision

gguf_formats = {
    'Q8_0':   {'bits': 8.0,  'size_7b_gb': 7.7,  'quality': 'Best',      'speed': 'Slow'},
    'Q6_K':   {'bits': 6.6,  'size_7b_gb': 6.1,  'quality': 'Excellent', 'speed': 'Medium'},
    'Q5_K_M': {'bits': 5.7,  'size_7b_gb': 5.2,  'quality': 'Very Good', 'speed': 'Medium'},
    'Q4_K_M': {'bits': 4.8,  'size_7b_gb': 4.4,  'quality': 'Good',      'speed': 'Fast'},  # sweet spot
    'Q4_K_S': {'bits': 4.4,  'size_7b_gb': 4.0,  'quality': 'Good',      'speed': 'Fast'},
    'Q3_K_M': {'bits': 3.9,  'size_7b_gb': 3.5,  'quality': 'Fair',      'speed': 'Very Fast'},
    'Q2_K':   {'bits': 3.35, 'size_7b_gb': 3.1,  'quality': 'Poor',      'speed': 'Very Fast'},
}

print(f'{'Format':12} {'Bits':6} {'7B size':10} {'Quality':12} {'Speed'}')
print('-' * 55)
for fmt, info in gguf_formats.items():
    marker = ' ← recommended' if fmt == 'Q4_K_M' else ''
    print(f'{fmt:12} {info["bits"]:6.1f} {info["size_7b_gb"]:6.1f} GB   {info["quality"]:12} {info["speed"]}{marker}')

## AWQ and GPTQ — GPU quantization

In [ ]:
# AWQ (Activation-aware Weight Quantization) — best quality 4-bit for GPU
# pip install autoawq
awq_example = '''
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_path = "meta-llama/Llama-3.2-1B"
quant_path = "./llama-1b-awq"

model = AutoAWQForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Quantize (requires calibration data)
model.quantize(
    tokenizer,
    quant_config={"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
)
model.save_quantized(quant_path)
'''

# Many pre-quantized models are already on HuggingFace Hub
# Search: "TheBloke/Llama" or "bartowski/Llama" for GGUF/AWQ/GPTQ versions

print('Pre-quantized models on Hub:')
print('  GGUF: search "TheBloke" or "bartowski" + model name')
print('  AWQ:  search "<model>-AWQ"')
print('  GPTQ: search "<model>-GPTQ"')
print()
print('Example: bartowski/Llama-3.2-1B-Instruct-GGUF')
print('         TheBloke/Mistral-7B-Instruct-v0.2-GGUF')

## Quantization decision guide

```
Running locally (CPU/Mac/consumer GPU)?
  → GGUF Q4_K_M via Ollama — easiest

Running on NVIDIA GPU for inference?
  → AWQ 4-bit (best quality)
  → GPTQ 4-bit (good, widely supported)

Fine-tuning on consumer GPU?
  → NF4 4-bit via bitsandbytes (QLoRA)

Production API serving?
  → FP16/BF16 with vLLM (see S50_02)
  → INT8 via vLLM for memory savings
```

Next: [S50_02_inference_servers.ipynb](./S50_02_inference_servers.ipynb)